In [2]:
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.patches import Rectangle, Circle
from ipywidgets import interact, IntSlider, FloatSlider
from ZTF_Pipeline import SingleFrame, PipelineConfig, LightCurveExtractor

# --- CONFIGURATION & EXTRACTION ---
Field, Filter, CCD, Quadrant = "664", "zr", "10", "4"
FCQZ = f'{Field}_c{CCD}_q{Quadrant}_{Filter}'
diff_files = sorted(glob.glob(f".\\ztf_data\\{FCQZ}\\diffimg\\*.fits"))

ra_target, dec_target = 127.448018, 33.906536
zoom_size = 50
r_aperture = 10.0

cfg = PipelineConfig()
diff_frames = [SingleFrame(p, cfg) for p in diff_files]
extractor = LightCurveExtractor(diff_frames)
target_x, target_y = diff_frames[0].wcs.world_to_pixel_values(ra_target, dec_target)

df_lc = extractor.extract_at(x=target_x, y=target_y, r=r_aperture)
df_lc['mjd_float'] = pd.to_numeric(df_lc['mjd'])
# Conversion MJD -> Datetime for plotting
df_lc['dt'] = pd.to_datetime(df_lc['mjd_float'], unit='D', origin='1858-11-17')





def update_dashboard(index, poly_degree, sigma_clip, trim_percent):
    # Sigma-Clip
    m, s = df_lc['flux'].mean(), df_lc['flux'].std()
    df_clean = df_lc[(df_lc['flux'] > m - sigma_clip*s) & 
                     (df_lc['flux'] < m + sigma_clip*s)].copy()

    # Fit 
    mjd_min, mjd_max = df_clean['mjd_float'].min(), df_clean['mjd_float'].max()
    mjd_range = mjd_max - mjd_min
    
    x_scaled = df_clean['mjd_float'] - mjd_min
    coeffs = np.polyfit(x_scaled, df_clean['flux'], poly_degree)
    poly_func = np.poly1d(coeffs)

    # Clipping of the fit range
    margin = (trim_percent / 100.0) * mjd_range / 2
    x_fit_min = mjd_min + margin
    x_fit_max = mjd_max - margin
    
    x_fit_points = np.linspace(x_fit_min, x_fit_max, 300)
    y_fit = poly_func(x_fit_points - mjd_min)
    dt_fit = pd.to_datetime(x_fit_points, unit='D', origin='1858-11-17')
 
    # Plots===================================================================================
    fig = plt.figure(figsize=(22, 6))
    gs = fig.add_gridspec(1, 3, width_ratios=[1, 1, 1.8])
    frame = diff_frames[index]


    # --- FULL FRAME ---
    ax0 = fig.add_subplot(gs[0])
    ax0.imshow(frame.data, origin='lower', cmap='viridis', vmin=-50, vmax=200)
    rect = Rectangle((target_x - zoom_size, target_y - zoom_size), 
                     zoom_size*2, zoom_size*2, edgecolor='red', facecolor='none', lw=1.5)
    ax0.add_patch(rect)
    ax0.set_title(f"Full Difference Frame (Image {index})")


    # --- TARGET ZOOM ---
    ax1 = fig.add_subplot(gs[1])
    y_idx, x_idx = int(target_y), int(target_x)
    crop = frame.data[y_idx-zoom_size:y_idx+zoom_size, x_idx-zoom_size:x_idx+zoom_size]
    ax1.imshow(crop, origin='lower', cmap='viridis', vmin=-50, vmax=400,
               extent=[target_x-zoom_size, target_x+zoom_size, target_y-zoom_size, target_y+zoom_size])
    ax1.add_patch(Circle((target_x, target_y), r_aperture, edgecolor='red', facecolor='none', lw=2))
    ax1.set_title(f"Zoom on Target (x={target_x:.1f}, y={target_y:.1f})")


    # --- LIGHT CURVE ---
    ax2 = fig.add_subplot(gs[2])

    df_outliers = df_lc[~df_lc.index.isin(df_clean.index)]
    ax2.scatter(df_outliers['dt'], df_outliers['flux'], color='gray', alpha=0.15, label='Outliers')
    ax2.errorbar(df_clean['dt'], df_clean['flux'], yerr=df_clean['flux_err'], 
                 fmt='o', color='darkblue', alpha=0.9, label='Data')
    
    ax2.plot(dt_fit, y_fit, color='orange', lw=3, label=f'Fit Degree {poly_degree}', linestyle='--', alpha=0.9)
    
    # Show current point
    current = df_lc.iloc[index]
    ax2.scatter(current['dt'], current['flux'], s=180, facecolors='none', edgecolors='red', lw=3, zorder=10)

    # Change x-axis to datetime format
    ax2.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))
    plt.setp(ax2.get_xticklabels(), rotation=35, ha='right')
    
    ax2.set_ylabel("Flux")
    ax2.set_title(f"Light Curve | MJD actuel: {current['mjd_float']:.3f}")
    ax2.legend(loc='upper right')
    ax2.grid(True, alpha=0.2)
    
    plt.tight_layout()
    plt.show()

# --- Slider ---
interact(
    update_dashboard, 
    index = IntSlider(min=0, max=len(diff_frames)-1, value=0, description='Image'),
    poly_degree = IntSlider(min=1, max=15, value=12, description='Degree Poly'),
    sigma_clip = FloatSlider(min=1.0, max=10.0, step=0.1, value=3.0, description='Sigma Rejet'),
    trim_percent = IntSlider(min=0, max=50, step=2, value=0, description='Troncature %')
)

interactive(children=(IntSlider(value=0, description='Image', max=62), IntSlider(value=12, description='Degree…

<function __main__.update_dashboard(index, poly_degree, sigma_clip, trim_percent)>

In [ ]:
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.patches import Rectangle, Circle
from ipywidgets import interact, IntSlider, FloatSlider
from alerce.core import Alerce

# --- 1. CONFIGURATION & LOCAL DATA ---
from ZTF_Pipeline import SingleFrame, PipelineConfig, LightCurveExtractor

Field, Filter, CCD, Quadrant = "664", "zr", "10", "4"
FCQZ = f'{Field}_c{CCD}_q{Quadrant}_{Filter}'
diff_files = sorted(glob.glob(f".\\ztf_data\\{FCQZ}\\diffimg\\*.fits"))

ra_target, dec_target = 127.448018, 33.906536
zoom_size = 50
r_aperture = 10.0
ZP_REF = 30.0  

cfg = PipelineConfig()
diff_frames = [SingleFrame(p, cfg) for p in diff_files]
extractor = LightCurveExtractor(diff_frames)
target_x, target_y = diff_frames[0].wcs.world_to_pixel_values(ra_target, dec_target)

df_lc = extractor.extract_at(x=target_x, y=target_y, r=r_aperture)
df_lc['mjd_float'] = pd.to_numeric(df_lc['mjd'])
df_lc['dt'] = pd.to_datetime(df_lc['mjd_float'], unit='D', origin='1858-11-17')

# --- Official ZTF Data Retrieval & Conversion (MAG -> FLUX) ---
client = Alerce()
oid = "ZTF17aadlxmv"

try:
    df_ztf_all = client.query_detections(oid, format="pandas")
    df_ztf_r = df_ztf_all[df_ztf_all['fid'] == 2].copy()
    df_ztf_r['flux_ofl'] = 10**(0.4 * (ZP_REF - df_ztf_r['magpsf'])) # Magnitude to Flux conversion based on ZP_REF
    df_ztf_r['flux_err_ofl'] = df_ztf_r['flux_ofl'] * (df_ztf_r['sigmapsf'] * (np.log(10)/2.5)) # Error propagation for flux
    
    df_ztf_r['dt'] = pd.to_datetime(df_ztf_r['mjd'], unit='D', origin='1858-11-17')
    print(f"Données ZTF converties en flux : {len(df_ztf_r)} points.")
except Exception as e:
    print(f"Erreur ALeRCE : {e}")
    df_ztf_r = pd.DataFrame()


def update_dashboard(index, poly_degree, sigma_clip, trim_percent):
    # outliers sigma-clip
    m, s = df_lc['flux'].mean(), df_lc['flux'].std()
    df_clean = df_lc[(df_lc['flux'] > m - sigma_clip*s) & 
                     (df_lc['flux'] < m + sigma_clip*s)].copy()

    # Polyfit
    mjd_min, mjd_max = df_clean['mjd_float'].min(), df_clean['mjd_float'].max()
    x_scaled = df_clean['mjd_float'] - mjd_min
    coeffs = np.polyfit(x_scaled, df_clean['flux'], poly_degree)
    poly_func = np.poly1d(coeffs)

    # Clipping of the fit range
    margin = (trim_percent / 100.0) * (mjd_max - mjd_min) / 2
    x_fit_pts = np.linspace(mjd_min + margin, mjd_max - margin, 300)
    y_fit = poly_func(x_fit_pts - mjd_min)
    dt_fit = pd.to_datetime(x_fit_pts, unit='D', origin='1858-11-17')

    # Plots
    fig = plt.figure(figsize=(22, 7))
    gs = fig.add_gridspec(1, 3, width_ratios=[1, 1, 2])
    frame = diff_frames[index]
    current = df_lc.iloc[index]
    date_str = current['dt'].strftime('%Y-%m-%d')

    # AX0 : Full Frame
    ax0 = fig.add_subplot(gs[0])
    ax0.imshow(frame.data, origin='lower', cmap='viridis', vmin=-20, vmax=150)
    ax0.add_patch(Rectangle((target_x-zoom_size, target_y-zoom_size), zoom_size*2, zoom_size*2, 
                            edgecolor='red', facecolor='none', lw=1.5))
    plt.colorbar(ax0.imshow(frame.data, origin='lower', cmap='viridis', vmin=0, vmax=100), ax=ax0, fraction=0.046, pad=0.04, label='ADU')
    ax0.set_title("Full Frame (CCD)")

    # AX1 : Zoom
    ax1 = fig.add_subplot(gs[1])
    y_i, x_i = int(target_y), int(target_x)
    crop = frame.data[y_i-zoom_size:y_i+zoom_size, x_i-zoom_size:x_i+zoom_size]
    ax1.imshow(crop, origin='lower', cmap='viridis', vmin=-50, vmax=400,
               extent=[target_x-zoom_size, target_x+zoom_size, target_y-zoom_size, target_y+zoom_size])
    ax1.add_patch(Circle((target_x, target_y), r_aperture, edgecolor='red', facecolor='none', lw=2))
    ax1.set_title(f"Zoom - {date_str}")
    
    ax2 = fig.add_subplot(gs[2])
    # ZTF Official Data
    if not df_ztf_r.empty:
        ax2.errorbar(df_ztf_r['dt'][:-1], df_ztf_r['flux_ofl'][:-1], yerr=df_ztf_r['flux_err_ofl'][:-1],
                     fmt='o', color='orange', alpha=0.7, label='ZTF Officiel Flux')

    # Local data
    ax2.errorbar(df_clean['dt'], df_clean['flux'], yerr=df_clean['flux_err'], 
                 fmt='o', color='darkblue', alpha=0.7, label='Local Flux')
    

    ax2.plot(dt_fit, y_fit, color='darkblue', lw=2.5, label=f'Fit Poly (deg={poly_degree})')    
    # Actual Point
    ax2.scatter(current['dt'], current['flux'], s=200, facecolors='none', edgecolors='red', lw=3, zorder=10)
    ax2.set_ylabel("Flux (Échelle commune ZP=30)")
    ax2.set_title(f"ZTF17aadlxmv en FLUX | {date_str}")
    ax2.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))
    plt.setp(ax2.get_xticklabels(), rotation=35, ha='right')
    ax2.legend()
    ax2.grid(True, alpha=0.2)
    
    plt.tight_layout()
    plt.show()

# --- Slider ---
interact(
    update_dashboard, 
    index = IntSlider(min=0, max=len(df_lc)-1, value=0),
    poly_degree = IntSlider(min=1, max=15, value=12),
    sigma_clip = FloatSlider(min=1.0, max=5.0, step=0.1, value=3.0),
    trim_percent = IntSlider(min=0, max=40, step=5, value=0)
)

Données ZTF converties en flux : 19 points.


interactive(children=(IntSlider(value=0, description='index', max=62), IntSlider(value=12, description='poly_d…

<function __main__.update_dashboard(index, poly_degree, sigma_clip, trim_percent)>